# Activiteit & Planning

**Thema:** De dynamiek van de agenda en vergaderzalen.

**Kernvragen:** Hoe vaak wordt er vergaderd, welke zalen zijn het populairst, hoe vaak worden activiteiten verplaatst of geannuleerd, en wat is de verdeling tussen openbare en besloten activiteiten?

Dit notebook verkent o.a. de silver-tabellen `activiteit`, `vergadering`, `zaal`, `reservering` en `activiteit_vervangen_vanuit`.

In [ ]:
# Importeer Path om met mappen en bestandspaden te werken
from pathlib import Path

# Importeer DuckDB voor de databaseverbinding
import duckdb

# Importeer Pandas voor tabellen en data-analyse
import pandas as pd

# Importeer Matplotlib voor grafieken
import matplotlib.pyplot as plt


# Zoek automatisch naar het databasebestand tweedekamer.duckdb
def find_database(start: Path | None = None) -> Path:
    
    # Begin in de huidige map
    current = (start or Path.cwd()).resolve()
    
    # Kijk in de huidige map en bovenliggende mappen
    for folder in (current, *current.parents):
        
        # Maak het verwachte pad naar het databasebestand
        candidate = folder / "tweedekamer.duckdb"
        
        # Controleer of het bestand bestaat
        if candidate.is_file():
            return candidate
    
    # Geef een foutmelding als de database niet gevonden wordt
    raise FileNotFoundError("Could not find tweedekamer.duckdb from the current directory")


# Zoek het databasebestand
database_path = find_database()

# Maak een alleen-lezen verbinding met de DuckDB-database
connection = duckdb.connect(str(database_path), read_only=True)


# Functie om een SQL-query uit te voeren en het resultaat als Pandas DataFrame terug te geven
def query_df(sql: str, parameters=None) -> pd.DataFrame:
    return connection.execute(sql, parameters or []).fetchdf()


# Laat zien waar het databasebestand gevonden is
database_path

## Onderwerp 1

In dit onderdeel onderzoek ik hoe vaak er wordt vergaderd en welke vergaderzalen het meest worden gebruikt.

In [ ]:
# Haal per zaal het aantal vergaderingen op
vergaderzalen = query_df("""
SELECT
    zaal,                              -- Naam van de zaal
    COUNT(*) AS aantal_vergaderingen  -- Tel het aantal vergaderingen
FROM silver.vergadering
WHERE zaal IS NOT NULL                 -- Negeer lege zaalnamen
GROUP BY zaal                          -- Groepeer per zaal
ORDER BY aantal_vergaderingen DESC     -- Hoogste aantal bovenaan
LIMIT 20                               -- Alleen de top 20
""")

# Laat het resultaat als tabel zien
vergaderzalen

In [ ]:
# Maak een horizontale staafgrafiek van het aantal vergaderingen per zaal
vergaderzalen.plot.barh(
    x="zaal",                    # Zaalnamen op de y-as
    y="aantal_vergaderingen",    # Aantal vergaderingen op de x-as
    legend=False                 # Geen legenda nodig
)

# Geef de grafiek een titel
plt.title("Top 20 vergaderzalen met de meeste vergaderingen", fontsize=16)

# Geef de assen duidelijke namen
plt.xlabel("Aantal vergaderingen", fontsize=12)
plt.ylabel("Vergaderzaal", fontsize=12)

# Zet de zaal met de meeste vergaderingen bovenaan
plt.gca().invert_yaxis()

# Zorg dat alles netjes binnen de grafiek past
plt.tight_layout()

# Toon de grafiek
plt.show()

Conclusie: De Plenaire zaal wordt het vaakst gebruikt, gevolgd door de Thorbeckezaal, Groen van Prinstererzaal en de Troelstrazaal. Deze zalen worden duidelijk vaker gebruikt dan de overige zalen.
Daarnaast valt op dat sommige zaalnamen mogelijk dubbel voorkomen door verschillende schrijfwijzen. Voorbeelden hiervan zijn Klompzaal en Klompézaal en Enquetezaal en Enquêtezaal. Dit kan invloed hebben op de tellingen en is daarom een aandachtspunt voor de datakwaliteit.

## Onderwerp 2
Opschonen dubbele zaalnamen

In de data komen enkele zaalnamen meerdere keren voor met een iets andere schrijfwijze, zoals Klompzaal en Klompézaal. Waarschijnlijk gaat het hier om dezelfde zaal. Daarom worden deze namen in de volgende analyse samengevoegd, zodat de tellingen beter vergelijkbaar zijn.

In [ ]:
# Importeer hulpmiddel om accenten uit tekst te kunnen halen
import unicodedata

# Maak een kopie van de oorspronkelijke tabel
vergaderzalen_schoon = vergaderzalen.copy()


# Functie om zaalnamen te normaliseren
def maak_sleutel(naam):
    # Zet tekst om naar kleine letters en verwijder accenten
    naam = unicodedata.normalize("NFKD", naam).casefold()

    # Verwijder spaties en andere tekens; alleen letters/cijfers blijven over
    naam = "".join(teken for teken in naam if teken.isalnum())

    return naam


# Maak voor elke zaalnaam een gestandaardiseerde sleutel
vergaderzalen_schoon["zaal_sleutel"] = (
    vergaderzalen_schoon["zaal"].apply(maak_sleutel)
)


# Voeg bekende verschillende schrijfwijzen van dezelfde zaal samen
vergaderzalen_schoon["zaal_sleutel"] = (
    vergaderzalen_schoon["zaal_sleutel"].replace({
        "klompzaal": "klompezaal",
        "wttewaallvanstoetwegenzaal": "wttewaalvanstoetwegenzaal"
    })
)


# Tel de vergaderingen opnieuw op per opgeschoonde zaalnaam
vergaderzalen_schoon = (
    vergaderzalen_schoon
    .groupby("zaal_sleutel", as_index=False)["aantal_vergaderingen"]
    .sum()
    .sort_values("aantal_vergaderingen", ascending=False)
)


# Zet de opgeschoonde sleutels weer om naar nette zaalnamen
vergaderzalen_schoon["zaal"] = vergaderzalen_schoon["zaal_sleutel"].replace({
    "plenairezaal": "Plenaire zaal",
    "thorbeckezaal": "Thorbeckezaal",
    "groenvanprinstererzaal": "Groen van Prinstererzaal",
    "troelstrazaal": "Troelstrazaal",
    "suzegroenewegzaal": "Suze Groenewegzaal",
    "wttewaalvanstoetwegenzaal": "Wttewaal van Stoetwegenzaal",
    "klompezaal": "Klompézaal",
    "enquetezaal": "Enquêtezaal",
    "alettajacobszaal": "Aletta Jacobszaal",
    "tilanuskamer": "Tilanuskamer",
    "vondelingkamer": "Vondelingkamer",
    "maxvanderstoelzaal": "Max van der Stoelzaal",
    "oudezaal": "Oude zaal",
    "k4zaal": "K4 zaal",
    "studiotolkgeberentaal": "Studio tolk Gebarentaal",
    "activiteitenkanaal": "Activiteitenkanaal",
    "plenairezaaltk": "Plenairezaal TK"
})


# Laat alleen de zaalnaam en het aantal vergaderingen zien
vergaderzalen_schoon[["zaal", "aantal_vergaderingen"]]

In [ ]:
# Maak de grafiek groter
plt.figure(figsize=(14, 8))

# Maak een horizontale staafgrafiek van de opgeschoonde zaalnamen
vergaderzalen_schoon.plot.barh(
    x="zaal",                    # Zaalnamen
    y="aantal_vergaderingen",    # Aantal vergaderingen per zaal
    legend=False                 # Geen legenda tonen
)

# Geef de grafiek een titel
plt.title("Vergaderzalen na samenvoegen van dubbele namen", fontsize=16)

# Geef de assen duidelijke namen
plt.xlabel("Aantal vergaderingen", fontsize=12)
plt.ylabel("Vergaderzaal", fontsize=12)

# Zet de zaal met de meeste vergaderingen bovenaan
plt.gca().invert_yaxis()

# Zorg dat alles netjes binnen de figuur past
plt.tight_layout()

# Toon de grafiek
plt.show()

Conclusie: Na het opschonen van de zaalnamen zijn verschillende schrijfwijzen samengevoegd. Hierdoor stijgt het aantal vergaderingen voor de Klompézaal naar 532, de Wttewaal van Stoetwegenzaal naar 490 en de Enquêtezaal naar 201. Dit geeft een betrouwbaarder beeld van het daadwerkelijke gebruik van de vergaderzalen.

## onderwerp 3
Vervangingsrelatie per jaar

In [ ]:
# Bekijk de eerste 20 vervangingsrelaties
query_df("""
SELECT *
FROM silver.activiteit_vervangen_vanuit
LIMIT 20
""")

In [ ]:
# Tel per jaar hoeveel vervangingsrelaties er zijn geregistreerd
vervangingen_per_jaar = query_df("""
SELECT
    YEAR(relatie_gewijzigd_op) AS jaar,          -- Haal het jaar uit de datum
    COUNT(*) AS aantal_vervangingen              -- Tel het aantal vervangingsrelaties
FROM silver.activiteit_vervangen_vanuit
GROUP BY jaar                                    -- Groepeer per jaar
ORDER BY jaar                                    -- Zet de jaren op volgorde
""")

# Laat het resultaat als tabel zien
vervangingen_per_jaar

In [ ]:
# Maak een staafgrafiek van het aantal vervangingsrelaties per jaar
ax = vervangingen_per_jaar.plot.bar(
    x="jaar",                     # Jaar op de x-as
    y="aantal_vervangingen",      # Aantal vervangingsrelaties op de y-as
    figsize=(10, 6),              # Grootte van de grafiek
    legend=False                  # Geen legenda tonen
)

# Geef de grafiek een titel
plt.title("Aantal vervangingsrelaties per jaar", fontsize=16)

# Geef de assen duidelijke namen
plt.xlabel("Jaar", fontsize=12)
plt.ylabel("Aantal vervangingsrelaties", fontsize=12)

# Zorg dat de jaartallen recht staan
plt.xticks(rotation=0)

# Zet het aantal boven elke balk
for container in ax.containers:
    ax.bar_label(container, padding=3, fontsize=11)

# Zorg dat alles netjes binnen de figuur past
plt.tight_layout()

# Toon de grafiek
plt.show()

Conclusie: Deze tabel laat zien wanneer een activiteit een eerdere activiteit vervangt. Dit kan bijvoorbeeld gebeuren bij een wijziging of verplaatsing. Uit deze tabel alleen is niet direct af te leiden of een activiteit daadwerkelijk is geannuleerd. In 2024 en 2026 zijn veel meer vervangingsrelaties geregistreerd dan in 2025. In 2024 zijn dit 2980 relaties en in 2026 2696, tegenover slechts 218 in 2025. Het lage aantal in 2025 valt sterk op en kan mogelijk wijzen op een verschil in registratie of onvolledige data. Daarom moet voorzichtig worden geconcludeerd dat er in 2025 daadwerkelijk veel minder activiteiten zijn vervangen.

## Onderwerp 4
Wat is de verdeling tussen openbare en besloten activiteiten?

In dit onderdeel onderzoek ik wat de verdeling is tussen openbare en besloten activiteiten.

In [ ]:
# Bekijk de eerste 5 activiteiten om te zien welke kolommen beschikbaar zijn
query_df("""
SELECT *
FROM silver.activiteit
LIMIT 5
""")

In [ ]:
# Tel hoeveel activiteiten openbaar, besloten of onbekend zijn
openbaar_besloten = query_df("""
SELECT
    CASE
        WHEN besloten = TRUE THEN 'Besloten'      -- True betekent besloten
        WHEN besloten = FALSE THEN 'Openbaar'     -- False betekent openbaar
        ELSE 'Onbekend'                           -- Voor lege/onbekende waarden
    END AS type_activiteit,
    COUNT(*) AS aantal                            -- Tel het aantal activiteiten
FROM silver.activiteit
GROUP BY type_activiteit                          -- Groepeer per type
ORDER BY aantal DESC                              -- Grootste aantal bovenaan
""")

# Laat de aantallen als tabel zien
openbaar_besloten

In [ ]:
# Maak een staafgrafiek van het aantal openbare en besloten activiteiten
ax = openbaar_besloten.plot.bar(
    x="type_activiteit",
    y="aantal",
    figsize=(8, 6),
    legend=False
)

# Geef de grafiek een titel
plt.title("Verdeling openbare en besloten activiteiten", fontsize=16)

# Geef de assen duidelijke namen
plt.xlabel("Type activiteit", fontsize=12)
plt.ylabel("Aantal activiteiten", fontsize=12)

# Zet de labels recht
plt.xticks(rotation=0)

# Zet het aantal boven elke balk
for container in ax.containers:
    ax.bar_label(container, padding=3, fontsize=11)

# Zorg dat alles netjes binnen de grafiek past
plt.tight_layout()

# Toon de grafiek
plt.show()

Conclusie: In de dataset komen meer openbare dan besloten activiteiten voor. Er zijn 46.652 openbare activiteiten en 26.591 besloten activiteiten geregistreerd. Daarmee is het grootste deel van de activiteiten openbaar.

## Controle

In [ ]:
# Controleer hoeveel vergaderingen er totaal zijn en hoeveel een zaal hebben
query_df("""
SELECT
    COUNT(*) AS totaal_vergaderingen,              -- Totaal aantal vergaderingen
    COUNT(zaal) AS vergaderingen_met_zaal,         -- Vergaderingen met ingevulde zaal
    COUNT(*) - COUNT(zaal) AS vergaderingen_zonder_zaal  -- Vergaderingen zonder zaal
FROM silver.vergadering
""")

Alle 7.507 vergaderingen hebben een zaal ingevuld. Er zijn dus geen ontbrekende zaalwaarden in deze tabel.